# Построение графов знаний на основе именованных сущностей

Граф знаний (Knowledge Graph) — это структура данных, которая представляет знания в виде сети взаимосвязанных сущностей. Это направленные реляционные графы, в которых сущности представлены в качестве вершин, а отношения между сущностями представлены как ребра. Для описания графов знаний используется набор триплетов, представленных как (субъект, отношение, объект). Такие графы обеспечивают структурированное представление фактов как об объектах реального мира, так и об абстрактных понятиях.

Узлы (nodes) → сущности (люди, компании, места и т.д.)

Рёбра (edges) → отношения между ними (например, "основал", "расположен в", "работает в").

Таким образом, NER обеспечивает основу для заполнения узлов графа, а дополнительные шаги (Relation Extraction) определяют рёбра между ними.

В этом задании мы построим граф персонажей романа "Преступление и наказание" Фёдора Михайловича Достоевского.

Ниже приведён список персонажей, чтобы вам было проще ориентироваться в полученных результатах.



**Персонажи**:

Родион Романович Раскольников – бывший студент юридического факультета, 23 года

Алена Ивановна – старуха-процентщица, вдова, коллежская секретарша, ок. 60 лет.

Семен Захарович Мармеладов – бывший чиновник, за 50 лет.

Софья Семеновна Мармеладова – дочь Мармеладова, ок. 18 лет.

Пульхерия Александровна Раскольникова – мать Раскольникова, 43 года.

Авдотья Романовна Раскольникова (Дуня) – младшая сестра Раскольникова

Катерина Ивановна Мармеладова – жена Мармеладова и мачеха Сони, ок. 30 лет, имеет трех детей.

Петр Петрович Лужин – надворный советник, человек с деньгами, 45 лет.

Дмитрий Прокофьевич Вразумихин (Разумихин) – молодой человек, студент, друг Раскольникова

Аркадий Иванович Свидригайлов – помещик, бывший шулер, ок. 50 лет.

Порфирий Петрович – следователь по делу об убийстве старухи-процентщицы и ее сестры, 35 лет.

**Второстепенные персонажи**:

Андрей Семенович Лебезятников – молодой человек, друг Лужина, придерживается прогрессивных взглядов

Лизавета Ивановна – сводная сестра старухи-процентщицы по отцу (у них были разные матери), 35 лет.

Марфа Петровна Свидригайлова – жена Свидригайлова, старше него на 5 лет

Зосимов – приятель Разумихина, знакомый Раскольникова, доктор, 27 лет.

Александр Григорьевич Заметов – знакомый Разумихина, письмоводитель в местной конторе, 22 года.

Никодим Фомич – квартальный надзиратель в том районе, где жил Раскольников

Илья Петрович – помощник квартального надзирателя по прозвищу «Порох»

Настасья – служанка в доме, где снимал комнату Раскольников

Амалия Ивановна – хозяйка в доме, где снимали комнату Мармеладовы

Миколка – красильщик, задержанный по делу об убийстве

In [ ]:
!pip install spacy
!python -m spacy download ru_core_news_sm

In [ ]:
%%capture
%pip install pyvis

In [ ]:
import chardet
import spacy
import pandas as pd
import numpy as np

import nltk
from nltk.tokenize import sent_tokenize
nltk.download('punkt_tab')

import pymorphy3

import networkx as nx

import matplotlib.pyplot as plt

from pyvis.network import Network
from IPython.display import display, HTML

import community.community_louvain as community_louvain

from spacy import displacy

Читаем из файла подготовленный текст романа.

In [ ]:
file_path = '/content/drive/MyDrive/Преступление_и_наказание.txt'

# Detect the encoding
with open(file_path, 'rb') as file:
    raw_data = file.read()
    result = chardet.detect(raw_data)
    encoding = result['encoding']

# Open the file with the detected encoding
with open(file_path, 'r', encoding=encoding) as file:
    data = file.read()

print(f"File opened successfully with encoding: {encoding}")
# You can now process the 'data' variable

In [ ]:
data

In [ ]:
lines = data.splitlines()

In [ ]:
len(lines)

Убираем пустые абзацы

In [ ]:
lines = list(filter(None, lines))

In [ ]:
lines[1]

In [ ]:
# Загрузка русской модели spacy для решения задачи NER (самой маленькой из доступных)
nlp = spacy.load("ru_core_news_sm")

In [ ]:


sent_entity_df = []

for sent in lines:
    doc = nlp(sent)
    entity_list = [ent.text for ent in doc.ents if ent.label_ == 'PER']
    if len(entity_list) > 0:
        sent_entity_df.append({"sentence": sent, "entities": entity_list})


In [ ]:
sent_entity_df = pd.DataFrame(sent_entity_df)

In [ ]:
# Визуализация сущностей в нескольких абзацах текста
doc = nlp(" ".join(lines[100:110]))
displacy.render(doc, style="ent", jupyter=True)

In [ ]:
sent_entity_df


In [ ]:
sent_entity_df.entities.value_counts()

In [ ]:
sent_entity_df['sentences'] = sent_entity_df['sentence'].apply(sent_tokenize)

In [ ]:
sent_entity_df

In [ ]:
sent_entity_df.loc[0].sentences

In [ ]:
sent_entity_df['num_sents'] = sent_entity_df['sentences'].apply(len)

In [ ]:
sent_entity_df[sent_entity_df.num_sents == 85].sentences

In [ ]:
sent_entity_df.iloc[1891].entities

Приводим найденные имена персонажей в именительный падеж там, где это возможно сделать с помощью морфологического анализатора pymorphy3

In [ ]:
# создаём морфологический анализатор
morph = pymorphy3.MorphAnalyzer()

def nominative(entities):
    names = []
    for word in entities:
        parse_results = morph.parse(word)
        parse_results = [p for p in parse_results if p.tag.POS == "NOUN"]  # отбираем только разборы как существительного
        if len(parse_results) > 0:
            name = parse_results[0].inflect({'nomn'})
            if name is not None:
                name = name.word
                names.append(name)
            else:
                names.append(word)
    return names

sent_entity_df['names'] = sent_entity_df['entities'].apply(nominative)

In [ ]:
sent_entity_df

In [ ]:
# Удаляем абзацы с пустыми списками персонажей и списками, в которых найден только один персонаж
sent_entity_df = sent_entity_df[sent_entity_df['names'].apply(lambda x: len(x) > 1)]
sent_entity_df.names.value_counts()

In [ ]:
sent_entity_df = sent_entity_df.reset_index()

In [ ]:
sent_entity_df

Сделаем отдельную таблицу для построения графа знаний. Для этого соберём все пары встречающихся в одном абзаце персонажей.

In [ ]:
relations_for_df = []

def get_character_pairs(names):
    relationships = []
    for i in range(len(names) - 1):
        for j in range(i + 1, len(names)):
            a = names[i]
            b = names[j]
            if a != b:
                relationships.append({"source": a, "target": b})
                relations_for_df.append({"source": a, "target": b})
    return relationships

In [ ]:
sent_entity_df['relationships'] = sent_entity_df['names'].apply(get_character_pairs)

In [ ]:
sent_entity_df.iloc[0].relationships

In [ ]:
sent_entity_df

Убираем строки, в которых мы не нашли пар персонажей.

In [ ]:
sent_entity_df = sent_entity_df[sent_entity_df['relationships'].apply(lambda x: len(x) > 0)].reset_index()


In [ ]:
sent_entity_df

Создаём новую таблицу из собранных данных

In [ ]:
relationship_df = pd.DataFrame(relations_for_df)

In [ ]:
relationship_df

Сортируем отношения

In [ ]:
relationship_df = pd.DataFrame(np.sort(relationship_df.values, axis = 1), columns = relationship_df.columns)
relationship_df

Считаем количества отношений

In [ ]:
relationship_df["value"] = 1
relationship_df = relationship_df.groupby(["source","target"], sort=False, as_index=False).sum()

In [ ]:
relationship_df

Оставим только те отношения, которые встречаются 15 и более раз, чтобы не перегружать граф.

In [ ]:
relationship_df = relationship_df[relationship_df['value'].apply(lambda x: x >= 15)].reset_index()

In [ ]:
relationship_df

Создаем граф из получившегося датафрейма

In [ ]:
G = nx.from_pandas_edgelist(relationship_df,
                            source = "source",
                            target = "target",
                            edge_attr = "value",
                            create_using = nx.Graph())

Для визуализации узлов в графе используем алгоритм Kamada-Kawai path-length cost-function - алгоритм для оптимизации расположения узлов в графе.

Алгоритм предназначен для минимизации суммы длин ребер графа, что важно для визуализации. Основная идея алгоритма заключается в том, чтобы найти оптимальное расположение узлов графа, минимизировать сумму длин ребер, принимая во внимание геометрические ограничения, такие как расстояние между узлами и угол между ребрами.

In [ ]:
plt.figure(figsize=(10,10))
pos = nx.kamada_kawai_layout(G)
nx.draw(G, with_labels=True, node_color='skyblue', edge_cmap=plt.cm.Blues, pos = pos)
plt.show()

Более красивую и интерактивную визуализацию можно получить с помощью библиотеки pyvis

In [ ]:
net = Network(notebook = True, width="1000px", height="700px", bgcolor='#222222', font_color='white', cdn_resources='remote')

node_degree = dict(G.degree)

#Setting up node size attribute
nx.set_node_attributes(G, node_degree, 'size')

net.from_nx(G)
display(HTML(net.generate_html()))

**Меры центральности узлов графа**

Мы можем посчитать центральность по степени (показывает, насколько важна конкретная вершина с точки зрения количества связей с другими вершинами в сети)

In [ ]:
degree_dict = nx.degree_centrality(G)
degree_dict

In [ ]:
degree_df = pd.DataFrame.from_dict(degree_dict, orient='index', columns=['centrality'])
# Plot top 10 nodes
degree_df.sort_values('centrality', ascending=False)[0:9].plot(kind="bar")

Также рассмотрим центральность по посредничеству - степень посредничества (данная мера основана на кратчайших путях, она показывает нам насколько часто рассматриваемая вершина i является «перевалочным пунктом» при переходах от одной вершины графа до любой другой)

In [ ]:
# Betweenness centrality
betweenness_dict = nx.betweenness_centrality(G)
betweenness_df = pd.DataFrame.from_dict(betweenness_dict, orient='index', columns=['centrality'])
# Plot top 10 nodes
betweenness_df.sort_values('centrality', ascending=False)[0:9].plot(kind="bar")

Центральность по близости (чем меньше расстояния от i-ой вершины до остальных j-ых вершин графа, тем больше будет значение самой центральности)

In [ ]:
# Closeness centrality
closeness_dict = nx.closeness_centrality(G)
closeness_df = pd.DataFrame.from_dict(closeness_dict, orient='index', columns=['centrality'])
# Plot top 10 nodes
closeness_df.sort_values('centrality', ascending=False)[0:9].plot(kind="bar")

Сохраняем полученные значения для кластеризации

In [ ]:
# Save centrality measures
nx.set_node_attributes(G, degree_dict, 'degree_centrality')
nx.set_node_attributes(G, betweenness_dict, 'betweenness_centrality')
nx.set_node_attributes(G, closeness_dict, 'closeness_centrality')

**Кластеризация графов и поиск сообществ**

Для поиска сообществ будем использовать алгоритм Louvain Community Detection

Это простой метод для извлечения структуры сообщества в сети. Эвристический метод, основанный на модульной оптимизации.

Алгоритм работает в 2 этапа. На первом шаге он назначает каждому узлу принадлежность к его собственному сообществу, а затем для каждого узла пытается найти максимальный положительный выигрыш в модульности, перемещая каждый узел во все соседние сообщества. Если положительный выигрыш не достигается, узел остается в своем первоначальном сообществе.

In [ ]:
communities = community_louvain.best_partition(G)

nx.set_node_attributes(G, communities, 'group')

In [ ]:
com_net = Network(notebook = True, width="1000px", height="700px", bgcolor='#222222', font_color='white', cdn_resources='remote')
com_net.from_nx(G)
display(HTML(com_net.generate_html()))

# Задания
1) Объедините разные разные представления персонажей в одно перед выделением и построением отношений в таблице. Например, "Дуня", "Дунёчки" и "Дунёчек" представьте как "Дуня".

2) Узнайте, как именно происходит склонение имён в pymorphy3 (на основании документации к pymorphy2). Почему мы могли получить представления "раскольников" и "раскольники" для Раскольникова?

3) Попробуйте построить граф персонажей для своей любимой книги.